# EUW Performance PCA

            This notebook builds the hourly performance PCA used later for PC
            periodograms and player-level projections. The focus is on what goes
            into the PCA, how much variance each component explains, and how the
            loading vectors should be read.

## Setup

            In Colab, the notebook mounts Google Drive and looks for the raw Riot
            Parquet at the same shared-drive path used by the other release
            notebooks: `/content/drive/Shareddrives/MSc_2026_Riot/db/riotData.parquet`.

            The notebook also needs the repository Python files. If they are not
            already present in the runtime or Drive, the setup cell tries to clone
            the release repository into `/content/MSc2026_LoL_Release`.

In [ ]:
# Local users should normally use the uv environment from README.md.
# This cell only installs missing packages when the notebook is opened in Colab.
import importlib.util
import subprocess
import sys

MODULE_TO_PACKAGE = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "duckdb": "duckdb",
    "astropy": "astropy",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "joblib": "joblib",
}

missing = [
    package
    for module, package in MODULE_TO_PACKAGE.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("Notebook packages are available.")

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore[import-not-found]  # noqa: F401
    except ImportError:
        return False
    return True


IN_COLAB = running_in_colab()
if IN_COLAB:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")


# Override this if your repository folder has a different Colab/Drive location.
ROOT_OVERRIDE = None
REPO_URL = "https://github.com/wadelab/MSc2026_LoL_Release.git"


def find_repo_root() -> Path | None:
    if ROOT_OVERRIDE is not None:
        candidate = Path(ROOT_OVERRIDE).expanduser()
        if (candidate / "riot_analysis.py").exists():
            return candidate.resolve()
        raise FileNotFoundError(f"ROOT_OVERRIDE does not contain riot_analysis.py: {candidate}")

    candidates = list(Path.cwd().resolve().parents)
    candidates.insert(0, Path.cwd().resolve())
    candidates.extend(
        [
            Path("/content/MSc2026_LoL_Release"),
            Path("/content/drive/MyDrive/MSc2026_LoL_Release"),
            Path("/content/drive/Shareddrives/MSc_2026_Riot/MSc2026_LoL_Release"),
        ]
    )
    for candidate in candidates:
        if (candidate / "riot_analysis.py").exists():
            return candidate.resolve()
    return None


ROOT = find_repo_root()
if ROOT is None and IN_COLAB:
    clone_target = Path("/content/MSc2026_LoL_Release")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_target)], check=True)
    ROOT = find_repo_root()

if ROOT is None:
    raise FileNotFoundError(
        "Could not find riot_analysis.py. Set ROOT_OVERRIDE to the repository folder."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 120

print(f"Repository root: {ROOT}")
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
from riot_analysis import (
                AnalysisConfig,
                GOOD_PCA_COLS,
                add_time_normalized_features,
                compute_pca,
                configure_plot_style,
                connect_analysis_database,
                filter_hourly_window,
                filter_metric_outliers,
                load_hourly_metrics,
                style_axes,
            )
            from server_timezones import hour_idx_to_local_hour_of_day

            configure_plot_style()

            PLATFORM = "EUW1"
            DB_FILE = ROOT / "riot_local.duckdb"
            PARQUET_FILE = None

            config = AnalysisConfig(platform=PLATFORM, max_hour_limit=5000, output_root=ROOT / "results")
            conn = connect_analysis_database(DB_FILE, parquet_file=PARQUET_FILE)

## Build time-normalized hourly features

            The raw counts are divided by mean `TIMEPLAYED` so PCA reflects
            performance rate rather than simply longer games.

In [ ]:
hourly_metrics = load_hourly_metrics(conn, PLATFORM)
            hourly_metrics = filter_hourly_window(hourly_metrics, config.max_hour_limit)
            hourly_metrics, numeric_cols = add_time_normalized_features(hourly_metrics)

            rows_before_outlier_filter = len(hourly_metrics)
            hourly_metrics = filter_metric_outliers(hourly_metrics, numeric_cols)

            print("PCA input columns:", numeric_cols)
            print(f"Rows before outlier filter: {rows_before_outlier_filter:,}")
            print(f"Rows after outlier filter:  {len(hourly_metrics):,}")
            display(hourly_metrics[["hour_idx", "n", *numeric_cols]].head())

## Run PCA and inspect component loadings

            The helper uses SVD on standardized features, then orients component
            signs so positive performance-rate variables tend to load positively.

In [ ]:
pca = compute_pca(hourly_metrics, numeric_cols, GOOD_PCA_COLS)
            loadings = pca["loadings"].iloc[:3].copy()
            explained = pd.DataFrame(
                {
                    "component": [f"PC{i + 1}" for i in range(6)],
                    "explained_variance": pca["explained"][:6],
                    "cumulative": np.cumsum(pca["explained"][:6]),
                }
            )

            pc_scores = hourly_metrics.loc[pca["features"].index, ["hour_idx"]].reset_index(drop=True)
            for i in range(3):
                pc_scores[f"PC{i + 1}"] = pca["scores"][:, i]
            pc_scores["hours_since_start"] = pc_scores["hour_idx"] - pc_scores["hour_idx"].min()
            pc_scores["local_hour"] = hour_idx_to_local_hour_of_day(pc_scores["hour_idx"], PLATFORM)

            display(explained)
            display(loadings)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

            ax = axes[0]
            ax.bar(explained["component"], explained["explained_variance"], color="#2a9d8f", alpha=0.9)
            ax.plot(explained["component"], explained["cumulative"], color="#1d3557", marker="o", linewidth=2.0, label="Cumulative")
            ax.set_title(f"PCA explained variance ({PLATFORM})")
            ax.set_ylabel("Fraction of variance")
            ax.legend(frameon=False)
            style_axes(ax, grid_axis="y")

            ax = axes[1]
            image = ax.imshow(loadings, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
            ax.set_title("First three PCA loading vectors")
            ax.set_yticks(range(len(loadings.index)))
            ax.set_yticklabels(loadings.index)
            ax.set_xticks(range(len(loadings.columns)))
            ax.set_xticklabels(loadings.columns, rotation=45, ha="right")
            fig.colorbar(image, ax=ax, label="Loading")

            fig.tight_layout()
            plt.show()

## Plot loadings and PC scores

            A loading is the weight assigned to a standardized input variable.
            A score is the value of that component for one hourly bin.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharex=False)
            for ax, component in zip(axes, loadings.index):
                ordered = loadings.loc[component].sort_values()
                colors = ["#e76f51" if value >= 0 else "#1d3557" for value in ordered]
                ax.barh(ordered.index, ordered.values, color=colors, alpha=0.9)
                ax.axvline(0, color="#264653", linewidth=1.0)
                ax.set_title(f"{component} loadings")
                ax.set_xlabel("Loading")
                style_axes(ax, grid_axis="x")
            fig.tight_layout()
            plt.show()

            fig, axes = plt.subplots(3, 1, figsize=(14, 8.5), sharex=True)
            for ax, component in zip(axes, ["PC1", "PC2", "PC3"]):
                ax.plot(pc_scores["hours_since_start"], pc_scores[component], color="#2a9d8f", linewidth=0.9, alpha=0.9)
                ax.axhline(0, color="#8d99ae", linestyle="--", linewidth=1.0)
                ax.set_title(f"{component} score over hourly bins")
                ax.set_ylabel("Score")
                style_axes(ax)
            axes[-1].set_xlabel("Hours since first retained bin")
            fig.tight_layout()
            plt.show()

## Local-hour PC profiles

            This is a descriptive fold of the hourly scores. It does not replace
            the periodogram, but it helps interpret a 24-hour signal if one is
            present.

In [ ]:
local_pc = pc_scores.groupby("local_hour", as_index=False)[["PC1", "PC2", "PC3"]].mean()
            local_pc = local_pc.set_index("local_hour").reindex(range(24)).reset_index()

            fig, ax = plt.subplots(figsize=(12, 4.8))
            for component, color in zip(["PC1", "PC2", "PC3"], ["#1d3557", "#2a9d8f", "#e76f51"]):
                ax.plot(local_pc["local_hour"], local_pc[component], marker="o", linewidth=2.0, label=component, color=color)
            ax.axhline(0, color="#8d99ae", linestyle="--", linewidth=1.0)
            ax.set_title(f"Mean PC score by local hour ({PLATFORM})")
            ax.set_xlabel("Local hour")
            ax.set_ylabel("Mean score")
            ax.set_xticks(range(0, 24, 2))
            ax.legend(frameon=False)
            style_axes(ax, grid_axis="y")
            plt.show()

            display(local_pc)
            conn.close()